In [ ]:
# Install necessary libraries
!pip install albumentations --user
!pip install ultralytics --user
!pip install flask --user
!pip install scikit-learn --user
!pip install torch --user
!pip install streamlit --user


In [2]:
# Libraries for Data Preprocessing
import os
import cv2
import albumentations as A
import numpy as np
import matplotlib.pyplot as plt

#Libraries for data splitting
import shutil
from sklearn.model_selection import train_test_split

# Libraries for ML Model
from ultralytics import YOLO
import torch

# Libraries for Python Web App
from flask import Flask, request, render_template, send_file

c:\Users\yongx\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
C:\Users\yongx\AppData\Roaming\Python\Python310\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 1.4.18 (you have 1.4.14). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [5]:
# Function for data splitting
def custom_train_test_split(dataset_directory, test_ratio=0.2, random_seed=42, min_samples_per_class=2):
    class_directories = [d for d in os.listdir(dataset_directory) if os.path.isdir(os.path.join(dataset_directory, d))]
    
    train_directory = os.path.join(dataset_directory, 'train')
    test_directory = os.path.join(dataset_directory, 'test')
    os.makedirs(train_directory, exist_ok=True)
    os.makedirs(test_directory, exist_ok=True)

    for class_dir in class_directories:
        class_path = os.path.join(dataset_directory, class_dir)
        image_files = [f for f in os.listdir(class_path) if f.endswith('.jpg')]

        if len(image_files) < min_samples_per_class:
            continue

        train_images, test_images = train_test_split(image_files, test_size=test_ratio, random_state=random_seed)

        for train_image in train_images:
            src_image_path = os.path.join(class_path, train_image)
            src_annotation_path = os.path.join(class_path, train_image.replace('.jpg', '.txt'))

            dest_image_path = os.path.join(train_directory, class_dir, train_image)
            dest_annotation_path = os.path.join(train_directory, class_dir, train_image.replace('.jpg', '.txt'))
            os.makedirs(os.path.dirname(dest_image_path), exist_ok=True)

            if os.path.exists(src_annotation_path):
                shutil.copy(src_image_path, dest_image_path)
                shutil.copy(src_annotation_path, dest_annotation_path)

        for test_image in test_images:
            src_image_path = os.path.join(class_path, test_image)
            src_annotation_path = os.path.join(class_path, test_image.replace('.jpg', '.txt'))

            dest_image_path = os.path.join(test_directory, class_dir, test_image)
            dest_annotation_path = os.path.join(test_directory, class_dir, test_image.replace('.jpg', '.txt'))
            os.makedirs(os.path.dirname(dest_image_path), exist_ok=True)

            if os.path.exists(src_annotation_path):
                shutil.copy(src_image_path, dest_image_path)
                shutil.copy(src_annotation_path, dest_annotation_path)

# Execute function for Data Splitting
dataset_directory = './Dataset Repository/Brain Tumor labeled dataset'
custom_train_test_split(dataset_directory, test_ratio=0.2, random_seed=42, min_samples_per_class=2)


In [6]:
# Define individual augmentation functions
def apply_transformations(image, bboxes, class_labels):
    transform_resize = A.Compose([A.Resize(640, 640)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
    transform_rotate = A.Compose([A.Resize(640, 640), A.RandomRotate90(p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
    transform_flip = A.Compose([A.Resize(640, 640), A.Flip(p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
    transform_transpose = A.Compose([A.Resize(640, 640), A.Transpose(p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
    transform_brightness_contrast = A.Compose([A.Resize(640, 640), A.RandomBrightnessContrast(p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
    transform_gauss_noise = A.Compose([A.Resize(640, 640), A.GaussNoise(var_limit=(10.0, 50.0), p=1.0)], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

    # Dictionary of transformations
    transformations = {
        'resize': transform_resize,
        'rotate': transform_rotate,
        'flip': transform_flip,
        'transpose': transform_transpose,
        'brightness_contrast': transform_brightness_contrast,
        'gauss_noise': transform_gauss_noise
    }

    # Apply transformations and store results
    augmented_images = {}
    for name, transform in transformations.items():
        augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
        augmented_images[name] = (augmented['image'], augmented['bboxes'], augmented['class_labels'])

    return augmented_images

# Function to save augmented images and annotations
def save_augmented_images(image_path, annotation_path, output_dir):
    # Load image and annotation
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    with open(annotation_path, 'r') as file:
        lines = file.readlines()

    bboxes = []
    class_labels = []
    for line in lines:
        class_number, centre_x, centre_y, width, height = map(float, line.strip().split())
        bboxes.append([centre_x, centre_y, width, height])
        class_labels.append(int(class_number))

    # Save the original image
    original_image_name = os.path.basename(image_path).replace('.jpg', '_original.jpg')
    original_image_path = os.path.join(output_dir, original_image_name)
    cv2.imwrite(original_image_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    
    # Save the original annotation
    original_annotation_path = original_image_path.replace('.jpg', '.txt')
    with open(original_annotation_path, 'w') as file:
        for bbox, label in zip(bboxes, class_labels):
            file.write(f"{label} {bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}\n")

    # Apply transformations
    augmented_images = apply_transformations(image, bboxes, class_labels)

    # Save each augmented image and corresponding annotation
    for aug_name, (aug_image, aug_bboxes, aug_labels) in augmented_images.items():
        aug_image_name = os.path.basename(image_path).replace('.jpg', f'_aug_{aug_name}.jpg')
        aug_image_path = os.path.join(output_dir, aug_image_name)
        cv2.imwrite(aug_image_path, cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR))

        # Save annotation
        aug_annotation_path = aug_image_path.replace('.jpg', '.txt')
        with open(aug_annotation_path, 'w') as file:
            for bbox, label in zip(aug_bboxes, aug_labels):
                file.write(f"{label} {bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}")


# Define paths and augment the train/test sets
dataset_directory = './Dataset Repository/Brain Tumor labeled dataset'
output_directory = './Dataset Repository/Augmented Brain Tumor labeled dataset'
os.makedirs(output_directory, exist_ok=True)

# Process train and test sets
for subfolder in ['train', 'test']:
    for class_name in ['glioma', 'meningioma', 'pituitary', 'notumor']:
        class_directory = os.path.join(dataset_directory, subfolder, class_name)
        aug_class_directory = os.path.join(output_directory, subfolder, class_name)
        os.makedirs(aug_class_directory, exist_ok=True)

        # Loop over each image in the class directory
        image_files = [f for f in os.listdir(class_directory) if f.endswith('.jpg')]
        for image_file in image_files:
            image_path = os.path.join(class_directory, image_file)
            annotation_path = image_path.replace('.jpg', '.txt')

            if os.path.exists(annotation_path):
                save_augmented_images(image_path, annotation_path, aug_class_directory)


In [ ]:
# Load and configure the YOLOv8 model
model = YOLO('yolov8s.yaml')

# Train the model using the data configuration file
model.train(
    data='./brain_tumor_dataset.yaml', 
    epochs=50, 
    imgsz=640, 
    batch=16, 
    project='./Best_Model'
)


In [4]:
# Load the model and validate
model = YOLO('C:/Users/yongx/Documents/GitHub/Brain-Tumour-Detection-and-Classification/Best_Model/train2/weights/best.pt')
results = model.val(data='C:/Users/yongx/Documents/GitHub/Brain-Tumour-Detection-and-Classification/brain_tumor_dataset.yaml')

# Access precision, recall, and mAP using the correct keys
precision = results.results_dict.get('metrics/precision(B)', None)
recall = results.results_dict.get('metrics/recall(B)', None)
map50 = results.results_dict.get('metrics/mAP50(B)', None)
map50_95 = results.results_dict.get('metrics/mAP50-95(B)', None)

# Calculate the F1 Score if precision and recall are available
if precision is not None and recall is not None:
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
else:
    f1_score = None
    print("Precision or Recall not found in results.")

# Print the evaluation metrics
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)
print("mAP@0.5:", map50)       
print("mAP@0.5-0.95:", map50_95)

Ultralytics YOLOv8.2.91  Python-3.10.9 torch-2.4.1+cpu CPU (Intel Core(TM) i7-10510U 1.80GHz)
YOLOv8s summary (fused): 168 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs


val: Scanning C:\Users\yongx\Documents\GitHub\Brain-Tumour-Detection-and-Classification\Dataset Repository\Augmented Brain Tumor labeled dataset\test\glioma... 3045 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3045/3045 [00:12<00:00, 236.27it/s]


val: New cache created: C:\Users\yongx\Documents\GitHub\Brain-Tumour-Detection-and-Classification\Dataset Repository\Augmented Brain Tumor labeled dataset\test\glioma.cache
WARNING  Box and segment counts should be equal, but got len(segments) = 18, len(boxes) = 3049. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 191/191 [30:00<00:00,  9.42s/it]


                   all       3045       3049      0.917      0.866      0.918      0.561
             pituitary        868        869      0.951       0.93      0.971      0.523
            meningioma        770        770       0.96      0.964      0.988      0.627
                glioma        637        640      0.805      0.666      0.739      0.364
               notumor        770        770      0.951      0.905      0.974      0.729
Speed: 3.6ms preprocess, 559.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs\detect\val6
Precision: 0.9167279584187039
Recall: 0.8661355765214561
F1 Score: 0.8907139365604577
mAP@0.5: 0.9181436336451365
mAP@0.5:0.95: 0.5607322484539423


This is just to test original bounding box on MRI image

In [7]:
# Function to draw a bounding box on an image
def draw_bounding_box(image_path, annotation_path):
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error loading image: {image_path}")
        return

    # Read the bounding boxes from the annotation file
    with open(annotation_path, 'r') as file:
        lines = file.readlines()

    # Image dimensions
    img_height, img_width = image.shape[:2]

    # Draw each bounding box on the image
    for line in lines:
        class_id, center_x, center_y, width, height = map(float, line.strip().split())

        # Convert YOLO format to rectangle coordinates
        x1 = int((center_x - width / 2) * img_width)
        y1 = int((center_y - height / 2) * img_height)
        x2 = int((center_x + width / 2) * img_width)
        y2 = int((center_y + height / 2) * img_height)

        # Draw the rectangle on the image
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Green box with thickness 2
        cv2.putText(image, f"Class {int(class_id)}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Display the image
    cv2.imshow("Image with Bounding Box", image)
    cv2.waitKey(0)  # Press any key to close the window
    cv2.destroyAllWindows()

# Paths to your image and corresponding annotation file
image_path = 'C:/Users/yongx/Documents/GitHub/Brain-Tumour-Detection-and-Classification/Dataset Repository/Brain Tumor labeled dataset/glioma/Tr-gl_0011.jpg'  # Replace with the path to your image
annotation_path = 'C:/Users/yongx/Documents/GitHub/Brain-Tumour-Detection-and-Classification/Dataset Repository/Brain Tumor labeled dataset/glioma/Tr-gl_0011.txt'  # Replace with the path to your annotation file

# Draw bounding box on the image
draw_bounding_box(image_path, annotation_path)
